# Continuous Maze: fixed global replay budget

One pool includes **recent training experience and the retained archive**. Similar goals share archive groups; novel goals receive space by thinning existing groups. At the group limit, the most similar existing pair is consolidated. SAC recomputes rewards and goal termination for shared transitions.

Use `global_replay`, then `temperature_carry` as a per-task-memory control with the **same recent buffer capacity**. Similar embeddings are a heuristic for reusable experience, not a guarantee of equal state-action coverage. See `docs/global-replay.md` for the policy and memory accounting.

Install the updated package in this kernel and restart it before running.

The default sequence contains **20 goals**, retaining the original four first and adding farthest-first free-cell centres using maze path distance. Set `GOAL_COUNT = 4` for the original sequence. The 100,000-slot pool and 16-group limit stay fixed. More tasks do not guarantee consolidation: similar goals may share existing groups. `SMOKE = True` still runs only two goals with small test budgets.

In [ ]:
import os
from pathlib import Path
from uuid import uuid4

import torch

from tbtrl.experiments.config import load_config
from tbtrl.experiments.diagnostics import NotebookDiagnostics
from tbtrl.experiments.goals import extend_maze_goals
from tbtrl.experiments.plots import plot_comparison, plot_run
from tbtrl.experiments.runner import run_experiment
from tbtrl.experiments.transfer import configure_transfer

root = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "configs/tbtrl_gridworldmaze_sac_factorised_actor.json").exists()
)
# global_replay | temperature_carry (per-task memory control)
EXPERIMENT = "global_replay"
ACTOR_VARIANT = "factorised_tbtrl"
GOAL_COUNT = 20  # use 4 to reproduce the original task sequence
GLOBAL_CAPACITY = 100_000
RECENT_CAPACITY = 50_000
MAX_GROUPS = 16
SMOKE = False
DEVICE = (
    "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
)
MODES = ("scratch", "sequential", "recovery")
config = configure_transfer(
    load_config(root / "configs/tbtrl_gridworldmaze_sac_factorised_actor.json"),
    EXPERIMENT,
    ACTOR_VARIANT,
)
config.goals = extend_maze_goals(config, GOAL_COUNT)
config.global_replay_capacity = GLOBAL_CAPACITY
config.recent_replay_capacity = RECENT_CAPACITY
config.replay_max_groups = MAX_GROUPS
# Match the current-task window and task sampling in both experimental conditions.
config.training["buffer_capacity"] = RECENT_CAPACITY
config.training["replay_tasks_per_batch"] = 4
if SMOKE:
    config = config.smoke()
config.validate()
config.name += f"-goals{len(config.goals)}"
output_root = Path(os.environ.get("TBTRL_OUTPUT_ROOT", str(root / "runs")))
output = output_root / f"{config.name}-memory-{uuid4().hex[:8]}"
print("SMOKE CHECK" if SMOKE else "FULL RESEARCH RUN")
print(f"Experiment={EXPERIMENT}; device={DEVICE}; seeds={config.seeds}")
print(f"Goals={len(config.goals)}; maximum archive groups={config.replay_max_groups}")
print("Ordered goals:", list(enumerate(config.goals)))
print(
    f"Replay mode={config.replay_mode}; global slots={config.global_replay_capacity:,}; recent slots={config.recent_replay_capacity:,}"
)
print(
    f"Steps/goal={config.training['total_steps']:,}; warmup={config.training['warmup_steps']:,}; recovery={config.recovery['total_steps']:,}"
)
get_ipython().run_line_magic("matplotlib", "inline")

In [ ]:
diagnostics = NotebookDiagnostics(rollout_episodes=config.eval_episodes)
records = run_experiment(
    config, output, device=DEVICE, modes=MODES, on_task_end=diagnostics, verbose=True
)
print(f"Saved to {output}")

In [ ]:
for row in records:
    memory = row.get("replay_memory")
    if memory:
        print(
            f"{row['mode']:10s} goal={row['task_id']} "
            f"recent={memory['recent_samples']:,} archive={memory['archive_samples']:,} "
            f"groups={memory['groups']} tasks={memory['tasks']} "
            f"allocated={memory['allocated_transition_bytes'] / 1_000_000:.3f} MB "
            f"steps={row['steps']:,}"
        )
        admission = memory["admission"]
        if row["mode"] == "sequential" and admission:
            similarity = admission["nearest_similarity"]
            similarity_text = "none (first group)" if similarity is None else f"{similarity:.4f}"
            print(
                f"  group={admission['group']}; nearest similarity={similarity_text}; "
                f"tail candidates={admission['tail_candidates']:,}; "
                f"admitted before quota={admission['admitted_candidates']:,}; "
                f"merged [kept, removed]={admission['merged_groups']}"
            )
            print("  Final group sizes:", memory["group_sizes"])
figure = plot_run(output)
figure

In [ ]:
comparison = plot_comparison(output)
comparison

Compare acquisition cost, all-goal retention, and recovery against the matched per-task control. Run several seeds and budget sizes before drawing conclusions. Logs and `metrics.json` include allocated transition bytes, occupancy, group sizes, admission similarity, and any merged pair. Independent recovery copies the same final archive for each goal; it is not a cyclic retraining sequence.

For a matched memory comparison, use the same `GOAL_COUNT` in both presets; the deterministic order is independent of the training seed and memory mode. The resolved coordinates are saved in the manifest and checkpoints. Twenty goals multiply the maximum interaction budget by five relative to four goals, and all-goal evaluation adds further cost. Full settings remain 500,000 steps per acquisition goal and 100,000 per recovery goal, with the existing early stopping rules. No full training is launched by changing this notebook.